In [64]:
import pandas as pd
import numpy  as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

from collections import Counter

import warnings
warnings.filterwarnings('ignore')

In [65]:
data = {
    "Étudier": ["Oui","Oui","Non","Non","Oui","Non","Oui","Oui","Non","Non",
                 "Oui","Oui","Non","Non","Oui","Oui","Non","Oui","Non","Oui"],
    "Dormir":  ["Oui","Non","Oui","Non","Oui","Oui","Non","Oui","Non","Oui",
                 "Oui","Non","Oui","Non","Oui","Oui","Non","Non","Oui","Oui"],
    "Temps":   ["Long","Court","Court","Long","Long","Long","Court","Court","Court","Long",
                 "Long","Long","Court","Long","Court","Long","Court","Court","Long","Long"],
    "Réussite":["Oui","Oui","Non","Non","Oui","Non","Oui","Oui","Non","Non",
                 "Oui","Oui","Non","Non","Oui","Oui","Non","Oui","Non","Oui"]
}

df = pd.DataFrame(data)

In [66]:
#Transfer à 0 et 1 :
df = df.replace({'Oui': 1, 'Non': 0})
df['Temps']= df['Temps'].astype('category')
df['Temps'] = df['Temps'].cat.codes
df.head()

,Étudier,Dormir,Temps,Réussite
0,1,1,1,1
1,1,0,0,1
2,0,1,0,0
3,0,0,1,0
4,1,1,1,1


Fonction entropie

In [67]:

def entropy(y):
    # works with strings or numbers
    counts = np.array(list(Counter(y).values()), dtype=float)
    ps = counts / counts.sum()
    return -np.sum(ps * np.log2(ps + 1e-12))  # tiny epsilon to avoid log(0)

Fonction Gain d'information

In [68]:
def gini(y):
    counts = np.array(list(Counter(y).values()), dtype=float)
    ps = counts / counts.sum()
    return 1.0 - np.sum(ps**2)

In [69]:
def is_numeric_array(a):
    return np.issubdtype(np.asarray(a).dtype, np.number)

In [70]:
def apply_split(X_column, threshold, y):
    X_column = np.asarray(X_column)
    y = np.asarray(y)

    if is_numeric_array(X_column):
        left_mask = X_column <= threshold
        right_mask = X_column > threshold
    else:
        # categorical split: equals vs not equals
        left_mask = (X_column == threshold)
        right_mask = (X_column != threshold)

    return y[left_mask], y[right_mask], left_mask, right_mask

In [71]:
def decision_tree(X, y, depth=0, max_depth=5, min_samples_split=2, criterion='entropy'):
    X = np.asarray(X, dtype=object)  # keep strings
    y = np.asarray(y, dtype=object)

    num_samples, num_features = X.shape
    unique_classes = np.unique(y)

    # stopping conditions
    if (len(unique_classes) == 1) or (depth >= max_depth) or (num_samples < min_samples_split):
        return Counter(y).most_common(1)[0][0]  # leaf prediction

    best_criterion_value = float('inf')
    best_feature_index = None
    best_threshold = None
    best_left_mask = None
    best_right_mask = None

    for feature_index in range(num_features):
        col = X[:, feature_index]
        thresholds = np.unique(col)

        for threshold in thresholds:
            y_left, y_right, left_mask, right_mask = apply_split(col, threshold, y)

            if len(y_left) == 0 or len(y_right) == 0:
                continue

            p_left = len(y_left) / num_samples
            p_right = len(y_right) / num_samples

            if criterion == 'entropy':
                current = p_left * entropy(y_left) + p_right * entropy(y_right)
            else:
                current = p_left * gini(y_left) + p_right * gini(y_right)

            if current < best_criterion_value:
                best_criterion_value = current
                best_feature_index = feature_index
                best_threshold = threshold
                best_left_mask = left_mask
                best_right_mask = right_mask

    if best_feature_index is None:
        return Counter(y).most_common(1)[0][0]

    left_subtree = decision_tree(X[best_left_mask], y[best_left_mask],
                                 depth + 1, max_depth, min_samples_split, criterion)
    right_subtree = decision_tree(X[best_right_mask], y[best_right_mask],
                                  depth + 1, max_depth, min_samples_split, criterion)

    # node = (feature, threshold, left_subtree, right_subtree)
    # interpretation:
    # - numeric: left is <= threshold, right is > threshold
    # - categorical: left is == threshold, right is != threshold
    return (best_feature_index, best_threshold, left_subtree, right_subtree)

In [72]:
#Fuite de données car la colonne 
# « Étudier » correspond exactement à la colonne cible « Réussite ».
X = df[["Dormir", "Temps"]].values   #  (text) Oui/Non و Long/Court
y = df["Réussite"].values

In [73]:
tree = decision_tree(X, y, max_depth=5, criterion="entropy")
print(tree)

(0, 0, (1, 0, 1, 0), (1, 0, 0, 1))


L'arbre comme règles

If Dormir = Non (0):
    If Temps ≤ 0 → Réussite = Oui (1)
    If Temps > 0 → Réussite = Non (0)

If Dormir = Oui (1):
    If Temps ≤ 0 → Réussite = Non (0)
    If Temps > 0 → Réussite = Oui (1)